###***Shahid Patel | 31 | 232A004***

***Experiment No.2: Implementing a multi-armed bandit problem and comparing different exploration strategies like epsilon-greedy and UCB.***

In [ ]:
# Shahid Patel | 31 | 232A004
# Random Selection
#importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#importing dataset
dataset=pd.read_csv('/content/Ads_Optimisation - Ads_Optimisation.csv')
dataset.head()
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Ad 1    10000 non-null  int64
 1   Ad 2    10000 non-null  int64
 2   Ad 3    10000 non-null  int64
 3   Ad 4    10000 non-null  int64
 4   Ad 5    10000 non-null  int64
 5   Ad 6    10000 non-null  int64
 6   Ad 7    10000 non-null  int64
 7   Ad 8    10000 non-null  int64
 8   Ad 9    10000 non-null  int64
 9   Ad 10   10000 non-null  int64
dtypes: int64(10)
memory usage: 781.4 KB


Approaching By Random Selection

In [ ]:
# Shahid Patel | 31 | 232A004

import random
N = 10000 # Number of rounds (users)
d = 10    # Number of ads

ads_selected = []
total_reward = 0

for n in range(N):
    ad = random.randrange(d)
    ads_selected.append(ad)
    reward = dataset.values[n, ad]
    total_reward += reward

print(f"Total reward for Random Approach: {total_reward}")

Total reward for Random Approach: 1239


Approaching BY Upper Confidence Bound

In [ ]:
# Shahid Patel | 31 | 232A004
import math
N = 10000 # Number of rounds (users)
d = 10    # Number of ads

ads_selected = []
numbers_of_selections = [0] * d
sums_of_rewards = [0] * d
total_reward = 0

for n in range(N):
    ad = -1
    max_upper_bound = 0
    for i in range(d):
        if (numbers_of_selections[i] > 0):
            average_reward = sums_of_rewards[i] / numbers_of_selections[i]
            delta_i = math.sqrt(2 * math.log(n + 1) / numbers_of_selections[i])
            upper_bound = average_reward + delta_i
        else:
            upper_bound = 1e400 # A very large number to ensure unselected ads are chosen first
        if upper_bound > max_upper_bound:
            max_upper_bound = upper_bound
            ad = i
    ads_selected.append(ad)
    numbers_of_selections[ad] += 1
    reward = dataset.values[n, ad]
    sums_of_rewards[ad] += reward
    total_reward += reward

print(f"Total reward for UCB Approach: {total_reward}")

Total reward for UCB Approach: 2125


In [ ]:
# Shahid Patel | 31 | 232A004
import pandas as pd
# Get the first 50 selected ads
first_50_ads = ads_selected[:50]

# Get the rewards for these first 50 selections
# We need the index 'n' (round number) to get the correct reward from the dataset
rewards_for_first_50 = [dataset.values[n, ad_id] for n, ad_id in enumerate(first_50_ads)]

# Create a DataFrame for easier analysis
df_first_50 = pd.DataFrame({
    'ad_id': first_50_ads,
    'reward': rewards_for_first_50
})

# Group by ad_id to get counts and sum of rewards
ad_summary = df_first_50.groupby('ad_id').agg(
    count=('ad_id', 'count'),
    total_reward=('reward', 'sum')
).reset_index()

# Sort by count in descending order
ad_summary_sorted = ad_summary.sort_values(by='count', ascending=False)

print("Summary of first 50 ads selected by UCB:")
display(ad_summary_sorted)

Summary of first 50 ads selected by UCB:


,ad_id,count,total_reward
7,7,7,2
0,0,6,1
8,8,6,1
6,6,6,1
1,1,5,0
2,2,4,0
5,5,4,0
4,4,4,0
3,3,4,0
9,9,4,0


**Conclusion A : The UCB approach gives higher total ad rewards henceforth we'll go with UCB approach.**

In [ ]:
# Shahid Patel | 31 | 232A004
import random
N = 10000 # Number of rounds (users)
d = 10    # Number of ads

# Initialize parameters for Beta distribution (alpha and beta)
# numbers_of_rewards_1[i] will be alpha_i (number of times ad i got reward 1)
# numbers_of_rewards_0[i] will be beta_i (number of times ad i got reward 0)
numbers_of_rewards_1 = [0] * d
numbers_of_rewards_0 = [0] * d

ads_selected_ts = []
total_reward_ts = 0

for n in range(N):
    ad = -1
    max_random = 0
    for i in range(d):
        # Draw a random sample from a Beta distribution for each ad
        # The parameters for beta distribution are (alpha + 1, beta + 1)
        # where alpha is numbers_of_rewards_1 and beta is numbers_of_rewards_0
        random_beta = random.betavariate(numbers_of_rewards_1[i] + 1, numbers_of_rewards_0[i] + 1)

        if random_beta > max_random:
            max_random = random_beta
            ad = i

    ads_selected_ts.append(ad)
    # Get the reward for the selected ad
    reward = dataset.values[n, ad]

    # Update the Beta distribution parameters based on the observed reward
    if reward == 1:
        numbers_of_rewards_1[ad] += 1
    else:
        numbers_of_rewards_0[ad] += 1

    total_reward_ts += reward

print(f"Total reward for Thompson Sampling: {total_reward_ts}")

Total reward for Thompson Sampling: 2607
